In [39]:
from preprocessing import preprocess_data
from feature_engineering import run_feature_engineering
from make_model import train_and_evaluate
from feature_importance import analyze_feature_importance
from inference import predict_score
from verify_training_data import diagnose_training_data

### Preprocessing
Raw data from multiple rides with the earable is combined into a master `.csv` per sensors, e.g. `master_gyroscope.csv`. Folders with raw data are stored in the same directory as this file, where the last digit character of the folder name is the distraction score.

In [40]:
preprocess_data()

Starting data preprocessing...
Saved Labels: ./processed/y_labels.csv (25 rides labeled)
Saved Sensor Data: ./processed/master_temperature_sensor.csv (38085 total rows across all rides)
Saved Sensor Data: ./processed/master_optical_temperature_sensor.csv (47954 total rows across all rides)
Saved Sensor Data: ./processed/master_barometer.csv (38085 total rows across all rides)
Saved Sensor Data: ./processed/master_accelerometer.csv (608821 total rows across all rides)
Saved Sensor Data: ./processed/master_photoplethysmography.csv (305214 total rows across all rides)
Saved Sensor Data: ./processed/master_magnetometer.csv (608769 total rows across all rides)
Saved Sensor Data: ./processed/master_gyroscope.csv (608803 total rows across all rides)

Preprocessing complete, data is ready for tsfresh feature extraction


### Feature engineering using tsfresh
The master `.csv` is processed by tsfresh to extract features such as maximum value, standard deviation, median, etc. which can be processed by XGBoost. Exclude sensors by putting them in the `exclude_sensor` list.

In [ ]:
# exclude_sensors = ["optical_temperature_sensor", "accelerometer", "gyroscope", "magnetometer", "barometer", "photoplethysmography", "temperature_sensor"]

# We suspect temperature gives a wrong indication due to the way we ordered the rides:
# The first ride was with high distraction score and the last ride with low distraction score. 
# So the temperature sensor is basically just learning to predict the order of the rides, which is not what we want.
exclude_sensors = ["optical_temperature_sensor", "temperature_sensor","magnetometer"]

run_feature_engineering(exclude_sensors=[])

Starting tsfresh feature engineering...
Extracting features for: barometer


Feature Extraction: 100%|██████████| 25/25 [00:00<00:00, 1944.40it/s]

Extracting features for: photoplethysmography



Feature Extraction: 100%|██████████| 100/100 [00:00<00:00, 1348.62it/s]


Extracting features for: optical_temperature_sensor


...

Extracting features for: gyroscope


Feature Extraction: 100%|██████████| 75/75 [00:00<00:00, 916.66it/s]


Extracting features for: accelerometer


Feature Extraction: 100%|██████████| 75/75 [00:00<00:00, 917.01it/s]


Merging data...
Total features kept for training: 160

Success, final dataset saved to: ./training_data/X_selected_features.csv


### Train and evaluate model using XGBoost

In [57]:
train_and_evaluate(custom_labels=False)

Dataset ready: 25 rides, 160 features per ride.
Evaluating model using Leave-One-Out Cross-Validation...

MODEL PERFORMANCE
Mean Absolute Error (MAE): 0.72
Root Mean Squared Error (RMSE): 0.86
The model's guesses are off by an average of 0.72 points on the 0-4 scale.

Training final production model on 100% of the data...
Trained model saved to: ./models/model.json


### Evaluate feature importance of model
XGBoost has built-in functions to show the importance per feature.

In [58]:
analyze_feature_importance()


Top 15 Most Important Features for Predicting Distraction:
----------------------------------------------------------------------
Rank  | Feature Name                                  | Importance Score
----------------------------------------------------------------------
1     | GYROSCOPE_Z__minimum                          | 0.2912
2     | GYROSCOPE_X__maximum                          | 0.1480
3     | GYROSCOPE_X__standard_deviation               | 0.0796
4     | ACCELEROMETER_Z__minimum                      | 0.0574
5     | GYROSCOPE_Z__standard_deviation               | 0.0553
6     | ACCELEROMETER_X__standard_deviation           | 0.0546
7     | GYROSCOPE_Y__standard_deviation               | 0.0451
8     | MAGNETOMETER_X__minimum                       | 0.0386
9     | PHOTOPLETHYSMOGRAPHY_RED__standard_deviation  | 0.0289
10    | PHOTOPLETHYSMOGRAPHY_AMBIENT__sum_values      | 0.0249
11    | GYROSCOPE_Y__minimum                          | 0.0154
12    | GYROSCOPE_Z__median     

### Run inference on a single ride
To run inference on a single ride (folder), give `new_ride_folder="..."` as an argument and the model will give a distraction score.

In [51]:
score = predict_score(new_ride_folder="demo_runs/OpenWearable_Recording_2026-03-26T101935.303786_3_demo")

Processing new ride: OpenWearable_Recording_2026-03-26T101935.303786_3_demo
Calculated distraction score: 2.97 / 4.0


### Check scores for all the rides in the training data
Gives a list of rides used in training the model and their scores according to the model

In [52]:
diagnose_training_data()

Loading data and model for diagnostics...

Ride ID (Shortened)       | True Score | Predicted  | Error     
-----------------------------------------------------------------
2026-03-26T101341.966067  | 4.0        | 3.80       | 0.20      
2026-03-26T102512.493306  | 2.0        | 2.04       | 0.04      
2026-03-26T103015.953397  | 1.0        | 1.01       | 0.01      
2026-03-26T103513.282691  | 0.0        | 0.27       | 0.27      
2026-03-26T105712.346697  | 4.0        | 3.59       | 0.41      
2026-03-26T110257.381741  | 3.0        | 2.95       | 0.05      
2026-03-26T110809.275003  | 2.0        | 2.07       | 0.07      
2026-03-26T111245.688277  | 1.0        | 1.08       | 0.08      
2026-03-26T111650.158656  | 0.0        | 0.53       | 0.53      
2026-03-26T112950.696222  | 4.0        | 3.66       | 0.34      
2026-03-26T113555.625881  | 3.0        | 3.12       | 0.12      
2026-04-13T164718.788978  | 2.0        | 1.95       | 0.05      
2026-04-13T165102.851616  | 1.0        | 1.26 

### Clear processed data and model
Clear data to show the preprocessing and model training from beginning to end

In [23]:
# clear folders:
import shutil
shutil.rmtree("processed")
shutil.rmtree("models")
shutil.rmtree("training_data")